# Download Trained Model from Azure ML

**Previous:** [02-submit-training-job.ipynb](02-submit-training-job.ipynb) | **Next:** [04-optimize-model.ipynb](04-optimize-model.ipynb)

---

This notebook downloads the fine-tuned model from Azure ML to your local environment.

## What This Notebook Does

1. **Lists Training Jobs**: Shows completed training jobs in your Azure ML workspace
2. **Selects Job**: Identifies the training job with the model you want to download
3. **Downloads Model Artifacts**: Pulls model weights, checkpoints, and metadata from Azure ML
4. **Verifies Files**: Ensures all model components downloaded correctly
5. **Prepares for Next Steps**: Stages model for optimization or evaluation

## Why Download the Model?

- **Local Testing**: Test the model on your machine before deployment
- **Optimization**: Optimize model for inference (quantization, ONNX export)
- **Evaluation**: Run comprehensive evaluation metrics locally
- **Deployment**: Package model for containerized deployment
- **Backup**: Keep a local copy of trained models

## Downloaded Artifacts

Models are downloaded to:
```
models/
  ├── trained/
  │   ├── <job-name>/
  │   │   ├── adapter_model.bin  # LoRA weights
  │   │   ├── adapter_config.json
  │   │   ├── training_args.json
  │   │   ├── tokenizer files
  │   │   └── checkpoints/ (optional)
```

## Prerequisites

- Completed remote training job (notebook 02-submit-training-job.ipynb)
- Training job finished successfully
- Sufficient local disk space (~5-20GB depending on model)
- Azure ML workspace access

## Expected Duration

~5-15 minutes depending on model size and network speed

## 1. Setup and Connect to Workspace

**Why use ModelDownloader?** The `ModelDownloader` class handles Azure ML SDK complexity for locating job outputs, downloading with progress tracking, validating checksums, and organizing files in the correct directory structure.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from src.training.model_downloader import ModelDownloader
import pandas as pd

# Initialize Azure ML client
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

print(f"✓ Connected to workspace: {ml_client.workspace_name}")

# Initialize model downloader
downloader = ModelDownloader(ml_client=ml_client)
print("✓ Model downloader initialized")

## 2. Find Your Training Job

**Why list jobs?** You need the job ID to download its outputs. This table shows recent training runs so you can select the correct one, especially if you've run multiple experiments with different hyperparameters.

In [ ]:
# List recent training jobs
experiment_name = "phi-4-training"  # Update if different

jobs = ml_client.jobs.list(max_results=10)
job_list = []

for job in jobs:
    if job.experiment_name == experiment_name:
        job_list.append({
            "Job ID": job.name,
            "Display Name": job.display_name,
            "Status": job.status,
            "Created": job.creation_context.created_at.strftime("%Y-%m-%d %H:%M"),
        })

if job_list:
    df = pd.DataFrame(job_list)
    print(f"Recent training jobs in '{experiment_name}':")
    print(df.to_string(index=False))
else:
    print(f"No jobs found in experiment: {experiment_name}")

In [ ]:
# Set your job ID here (from the table above)
job_name = "<paste-job-id-here>"  # e.g., "kind_drum_abc123"

# Verify job exists and is complete
try:
    job = ml_client.jobs.get(job_name)
    print(f"Job: {job.name}")
    print(f"Status: {job.status}")
    print(f"Studio URL: {job.studio_url}")

    if job.status != "Completed":
        print(f"\n⚠️  Warning: Job status is '{job.status}', not 'Completed'")
        print("Model download may fail or be incomplete.")
except Exception as e:
    print(f"❌ Job not found: {e}")
    print("Please check the job ID and try again.")

## 3. View Training Metrics

In [ ]:
# Get job metrics from MLflow
from src.training.job_manager import AzureMLJobManager
from src.utils.config import load_config

config = load_config()
job_manager = AzureMLJobManager(
    subscription_id=config.azure.subscription_id,
    resource_group=config.azure.resource_group,
    workspace_name=config.azure.workspace_name,
)

print("Fetching training metrics...")
metrics = job_manager.get_job_metrics(job_name)

if metrics:
    print("\nTraining Metrics:")
    print("=" * 50)
    for key, value in metrics.items():
        print(f"  {key}: {value}")
    print("=" * 50)
else:
    print("⚠️  No metrics available. Job may not have logged metrics.")

## 4. List Available Checkpoints

In [ ]:
# List checkpoints saved during training
print("Listing checkpoints...")
checkpoints = downloader.list_job_checkpoints(job_name)

if checkpoints:
    print(f"\nFound {len(checkpoints)} checkpoints:")
    for i, checkpoint in enumerate(checkpoints, 1):
        print(f"  {i}. {checkpoint}")
else:
    print("⚠️  No checkpoints found in job outputs")

In [ ]:
# Identify best checkpoint
best_checkpoint = downloader.get_best_checkpoint(job_name)
print(f"\n✓ Best checkpoint identified: {best_checkpoint}")

## 5. Download Model Checkpoint

In [ ]:
# Configure download
output_path = project_root / "models" / "trained" / job_name
checkpoint_to_download = best_checkpoint  # Or choose specific checkpoint

print(f"Downloading checkpoint: {checkpoint_to_download}")
print(f"Output path: {output_path}")
print("\nThis may take several minutes for large models...\n")

# Download model
model_path = downloader.download_from_job(
    job_name=job_name,
    output_path=str(output_path),
    checkpoint=checkpoint_to_download,
)

print(f"\n✓ Model downloaded to: {model_path}")

## 6. Validate Model Files

In [ ]:
# Validate downloaded files
validation_results = downloader.validate_model_files(model_path)

print("\nModel File Validation:")
print("=" * 50)

all_valid = True
for file, present in validation_results.items():
    status = "✓ Present" if present else "✗ Missing"
    print(f"{status:12} - {file}")
    if not present:
        all_valid = False

print("=" * 50)
if all_valid:
    print("✓ All required files present")
else:
    print("⚠️  Some files are missing - model may not load correctly")

## 7. Extract Model Metadata

In [ ]:
# Extract and display metadata
metadata = downloader.extract_model_metadata(model_path)

print("\nModel Metadata:")
print("=" * 50)
print(f"Path: {metadata['path']}")
print(f"Size: {metadata['size_mb']:.1f} MB")
print(f"Model Type: {metadata['model_type']}")
print(f"Vocabulary Size: {metadata['vocab_size']:,}")
print(f"Hidden Size: {metadata['hidden_size']:,}")
print(f"Number of Layers: {metadata['num_layers']}")
print("=" * 50)

In [ ]:
# List all files in model directory
print("\nModel Files:")
for file_path in sorted(model_path.rglob("*")):
    if file_path.is_file():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        rel_path = file_path.relative_to(model_path)
        print(f"  {rel_path} ({size_mb:.1f} MB)")

## 8. Test Model Loading

In [ ]:
# Test loading the model with transformers
print("Testing model loading...")

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    print(f"✓ Tokenizer loaded: {len(tokenizer)} tokens")

    print("\nLoading model...")
    print("Note: This may take a few minutes for large models")

    # For LoRA models, load with PEFT
    adapter_config = model_path / "adapter_config.json"
    if adapter_config.exists():
        print("Detected LoRA adapter - loading with PEFT")
        from peft import PeftModel, PeftConfig

        # Need to load base model first
        print("⚠️  LoRA model requires base model for loading")
        print("   Base model path needed for inference")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            device_map="cpu",  # Use CPU for testing
            low_cpu_mem_usage=True,
        )
        print(f"✓ Model loaded successfully")
        print(f"   Parameters: {model.num_parameters():,}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\nModel files may be incomplete or corrupted.")

## 9. Register in Azure ML Model Registry

In [ ]:
# Register model in Azure ML for easy versioning and deployment
print("Registering model in Azure ML Model Registry...")

model_name = "phi-4-finetuned"

print(f"Registering model: {model_name}")
registered_model = job_manager.register_model_from_job(
    job_name=job_name,
    model_name=model_name,
    model_path=f"outputs/{checkpoint_to_download}",
    description="Phi-4 fine-tuned with LoRA",
    tags={
        "framework": "pytorch",
        "task": "text-generation",
        "base_model": "phi-4",
        "training_method": "lora",
        "job_id": job_name,
    },
)

print(f"\n✓ Model registered successfully!")
print(f"  Name: {registered_model.name}")
print(f"  Version: {registered_model.version}")
print(f"  ID: {registered_model.id}")

## Summary

✅ **Trained model successfully downloaded from Azure ML!**

**What was downloaded:**
- Fine-tuned model adapter (LoRA weights)
- Model configuration files
- Tokenizer files
- Training metadata

**Next steps:**
- Optimize model for inference (notebook 04-optimize-model.ipynb)
- Evaluate model performance (notebook 05-evaluate-model.ipynb)
- Deploy model (notebooks 06-07)

## Troubleshooting

**Job Not Found**:
- Verify job completed successfully in Azure ML Studio
- Check job name matches what you submitted
- Ensure you're connected to correct workspace

**Download Failures**:
- Check network connectivity
- Verify sufficient disk space
- Check Azure ML permissions
- Try downloading from Azure ML Studio as backup

**Missing Files**:
- Verify training job saved outputs correctly
- Check job logs for training completion
- Ensure model checkpointing was enabled during training

---

## Navigation

**Previous:** [02-submit-training-job.ipynb](02-submit-training-job.ipynb) | **Next:** [04-optimize-model.ipynb](04-optimize-model.ipynb)